In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import kurtosis, skew
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.distance import cdist

In [6]:
data_path = "../saved_models/STCRL/embeddings/embedded_trajectories.csv"
df = pd.read_csv(data_path)
df_human = pd.read_csv("../Dataset/Scores/human_scores.csv")

In [7]:
# df_human.columns
df_human = df_human[['exploration_score', 'exploitation_score',
       'exploration_exploitation_ratio']]
merged_df = pd.merge(df, df_human, left_index=True, right_index=True)
merged_df.columns

Index(['participant_id', 'session_no', 'task_type', 'trial_no', 'day', 'block',
       'start_point_x', 'start_point_y', 'target_point_x', 'target_point_y',
       'start_time', 'end_time', 'quadrant', 'is_success', 'actual_dist',
       'movement_dist', 'completion_time', 'path', 'time_string',
       'time_diff_ms', 'Age', 'Cohort', 'Gestational_Age',
       'mabc_total_test_score', 'mabc_standard_score', 'mabc_percentile',
       'distances', 'rmsd', 'normalized_trajectory',
       'trajectory_embedding_completion_time',
       'trajectory_embedding_sequential', 'trajectory_embedding_rmsd',
       'trajectory_embedding_success', 'trajectory_embedding_multi',
       'trajectory_embedding_weighted_multi', 'exploration_score',
       'exploitation_score', 'exploration_exploitation_ratio'],
      dtype='object')

In [9]:
merged_df.columns

Index(['participant_id', 'session_no', 'task_type', 'trial_no', 'day', 'block',
       'start_point_x', 'start_point_y', 'target_point_x', 'target_point_y',
       'start_time', 'end_time', 'quadrant', 'is_success', 'actual_dist',
       'movement_dist', 'completion_time', 'path', 'time_string',
       'time_diff_ms', 'Age', 'Cohort', 'Gestational_Age',
       'mabc_total_test_score', 'mabc_standard_score', 'mabc_percentile',
       'distances', 'rmsd', 'normalized_trajectory',
       'trajectory_embedding_completion_time',
       'trajectory_embedding_sequential', 'trajectory_embedding_rmsd',
       'trajectory_embedding_success', 'trajectory_embedding_multi',
       'trajectory_embedding_weighted_multi', 'exploration_score',
       'exploitation_score', 'exploration_exploitation_ratio'],
      dtype='object')

In [10]:
import numpy as np
import pandas as pd
from scipy import stats
import warnings

def calculate_trajectory_statistics(df, participant_id=None, session_no=None, task_type=None):
    """
    Calculate comprehensive trajectory performance statistics with 95% confidence intervals.
    
    Args:
        df: DataFrame with trajectory data
        participant_id: Specific participant ID (None for all participants)
        session_no: Specific session number (None for all sessions)
        task_type: Task type - "Unimanual" (0) or "Bimanual" (1), or None for both
    
    Returns:
        Dictionary containing statistics with mean ± 95% CI for each metric
    """
    
    # Create a copy of the dataframe to avoid modifying the original
    data = df.copy()
    
    # Apply filters
    filter_description = []
    
    if participant_id is not None:
        data = data[data['participant_id'] == participant_id]
        filter_description.append(f"Participant: {participant_id}")
    
    if session_no is not None:
        data = data[data['session_no'] == session_no]
        filter_description.append(f"Session: {session_no}")
    
    if task_type is not None:
        if isinstance(task_type, str):
            # Convert string to numeric
            task_numeric = 0 if task_type.lower() == "unimanual" else 1
        else:
            task_numeric = task_type
        data = data[data['task_type'] == task_numeric]
        task_name = "Unimanual" if task_numeric == 0 else "Bimanual"
        filter_description.append(f"Task: {task_name}")
    
    # Check if we have data after filtering
    if len(data) == 0:
        return {
            'error': 'No data found matching the specified criteria',
            'filters_applied': filter_description,
            'n_trials': 0
        }
    
    def calculate_mean_ci(values, confidence=0.95):
        """Calculate mean and 95% confidence interval for a series of values"""
        values = np.array(values)
        # Remove NaN values
        values = values[~np.isnan(values)]
        
        if len(values) == 0:
            return {'mean': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan, 'n': 0}
        
        mean_val = np.mean(values)
        
        if len(values) == 1:
            # Single value - no confidence interval
            return {'mean': mean_val, 'ci_lower': mean_val, 'ci_upper': mean_val, 'n': 1}
        
        # Calculate standard error
        sem = stats.sem(values)  # Standard error of the mean
        
        # Calculate 95% confidence interval using t-distribution
        t_value = stats.t.ppf((1 + confidence) / 2, len(values) - 1)
        margin_error = t_value * sem
        
        return {
            'mean': mean_val,
            'ci_lower': mean_val - margin_error,
            'ci_upper': mean_val + margin_error,
            'n': len(values)
        }
    
    def format_stat(stat_dict):
        """Format statistics for display"""
        if stat_dict['n'] == 0:
            return "No data"
        elif stat_dict['n'] == 1:
            return f"{stat_dict['mean']:.4f} (n=1)"
        else:
            return f"{stat_dict['mean']:.4f} ± {(stat_dict['ci_upper'] - stat_dict['mean']):.4f} (95% CI: {stat_dict['ci_lower']:.4f}-{stat_dict['ci_upper']:.4f})"
    
    # Calculate statistics for each metric
    results = {
        'filters_applied': filter_description,
        'n_trials': len(data),
        'statistics': {}
    }
    
    # 1. Exploration Score Statistics
    exploration_stats = calculate_mean_ci(data['exploration_score'])
    results['statistics']['exploration_score'] = {
        'raw': exploration_stats,
        'formatted': format_stat(exploration_stats)
    }
    
    # 2. Exploitation Score Statistics  
    exploitation_stats = calculate_mean_ci(data['exploitation_score'])
    results['statistics']['exploitation_score'] = {
        'raw': exploitation_stats,
        'formatted': format_stat(exploitation_stats)
    }
    
    # 3. Exploration-Exploitation Ratio Statistics
    ratio_stats = calculate_mean_ci(data['exploration_exploitation_ratio'])
    results['statistics']['exploration_exploitation_ratio'] = {
        'raw': ratio_stats,
        'formatted': format_stat(ratio_stats)
    }
    
    # 4. Success Rate (percentage)
    success_data = data['is_success']
    total_trials = len(success_data)
    successful_trials = np.sum(success_data == 1)
    success_rate = (successful_trials / total_trials) * 100 if total_trials > 0 else 0
    
    # Calculate 95% CI for success rate using binomial distribution
    if total_trials > 0 and successful_trials > 0 and successful_trials < total_trials:
        # Wilson score interval for binomial proportion
        z = stats.norm.ppf(0.975)  # 97.5th percentile for 95% CI
        p = successful_trials / total_trials
        n = total_trials
        
        denominator = 1 + (z**2 / n)
        center = (p + (z**2 / (2*n))) / denominator
        margin = z * np.sqrt((p*(1-p) + z**2/(4*n)) / n) / denominator
        
        success_ci_lower = max(0, (center - margin) * 100)
        success_ci_upper = min(100, (center + margin) * 100)
    else:
        # Edge cases: no trials, all success, or all failure
        success_ci_lower = success_rate
        success_ci_upper = success_rate
    
    results['statistics']['success_rate'] = {
        'raw': {
            'rate': success_rate,
            'ci_lower': success_ci_lower,
            'ci_upper': success_ci_upper,
            'successful_trials': successful_trials,
            'total_trials': total_trials
        },
        'formatted': f"{success_rate:.2f}% (95% CI: {success_ci_lower:.2f}%-{success_ci_upper:.2f}%) [{successful_trials}/{total_trials}]"
    }
    
    # 5. Completion Time Statistics
    completion_time_stats = calculate_mean_ci(data['completion_time'])
    results['statistics']['completion_time'] = {
        'raw': completion_time_stats,
        'formatted': format_stat(completion_time_stats).replace('.4f', '.2f')  # Use 2 decimal places for time
    }
    
    # 6. RMSD Statistics
    rmsd_stats = calculate_mean_ci(data['rmsd'])
    results['statistics']['rmsd'] = {
        'raw': rmsd_stats,
        'formatted': format_stat(rmsd_stats)
    }
    
    return results

def print_trajectory_statistics(df, participant_id=None, session_no=None, task_type=None):
    """
    Print formatted trajectory statistics in a readable format.
    """
    results = calculate_trajectory_statistics(df, participant_id, session_no, task_type)
    
    if 'error' in results:
        print(f"Error: {results['error']}")
        print(f"Filters applied: {', '.join(results['filters_applied']) if results['filters_applied'] else 'None'}")
        return results
    
    print("=" * 80)
    print("TRAJECTORY PERFORMANCE STATISTICS")
    print("=" * 80)
    
    # Print filter information
    if results['filters_applied']:
        print(f"Filters applied: {', '.join(results['filters_applied'])}")
    else:
        print("Filters applied: None (All data)")
    
    print(f"Total trials analyzed: {results['n_trials']}")
    print()
    
    # Print each statistic
    stats = results['statistics']
    
    print("EXPLORATION & EXPLOITATION METRICS:")
    print("-" * 40)
    print(f"Exploration Score:     {stats['exploration_score']['formatted']}")
    print(f"Exploitation Score:    {stats['exploitation_score']['formatted']}")
    print(f"Exploration/Exploitation Ratio: {stats['exploration_exploitation_ratio']['formatted']}")
    print()
    
    print("PERFORMANCE METRICS:")
    print("-" * 40)
    print(f"Success Rate:          {stats['success_rate']['formatted']}")
    print(f"Mean Completion Time:  {stats['completion_time']['formatted']} ms")
    print(f"Mean RMSD:             {stats['rmsd']['formatted']}")
    print()
    
    return results

# Example usage functions for common scenarios:

def analyze_participant_session_task(df, participant_id, session_no, task_type):
    """Analyze specific participant, session, and task combination"""
    return print_trajectory_statistics(df, participant_id=participant_id, 
                                     session_no=session_no, task_type=task_type)

def analyze_participant_all_sessions(df, participant_id, task_type=None):
    """Analyze specific participant across all sessions"""
    return print_trajectory_statistics(df, participant_id=participant_id, task_type=task_type)

def analyze_session_all_participants(df, session_no, task_type=None):
    """Analyze specific session across all participants"""
    return print_trajectory_statistics(df, session_no=session_no, task_type=task_type)

def analyze_task_all_data(df, task_type):
    """Analyze specific task type across all participants and sessions"""
    return print_trajectory_statistics(df, task_type=task_type)

def analyze_all_data(df):
    """Analyze all data without filters"""
    return print_trajectory_statistics(df)


In [11]:
results = analyze_participant_session_task(merged_df, participant_id="MTRLRN015", session_no=5, task_type="Bimanual")
# results = analyze_participant_all_sessions(df_with_ratio_progress, 'MTRLRN002')
# results = analyze_task_all_data(df_with_ratio_progress, 'Bimanual')
# results = analyze_all_data(df_with_ratio_progress)

TRAJECTORY PERFORMANCE STATISTICS
Filters applied: Participant: MTRLRN015, Session: 5, Task: Bimanual
Total trials analyzed: 24

EXPLORATION & EXPLOITATION METRICS:
----------------------------------------
Exploration Score:     0.0271 ± 0.0051 (95% CI: 0.0221-0.0322)
Exploitation Score:    0.4803 ± 0.0631 (95% CI: 0.4172-0.5434)
Exploration/Exploitation Ratio: 0.0681 ± 0.0230 (95% CI: 0.0451-0.0910)

PERFORMANCE METRICS:
----------------------------------------
Success Rate:          100.00% (95% CI: 100.00%-100.00%) [24/24]
Mean Completion Time:  5.0968 ± 0.6407 (95% CI: 4.4560-5.7375) ms
Mean RMSD:             152.8541 ± 15.4856 (95% CI: 137.3685-168.3396)



In [12]:
results = analyze_participant_session_task(merged_df, participant_id="MTRLRN015", session_no=1, task_type="Bimanual")


TRAJECTORY PERFORMANCE STATISTICS
Filters applied: Participant: MTRLRN015, Session: 1, Task: Bimanual
Total trials analyzed: 24

EXPLORATION & EXPLOITATION METRICS:
----------------------------------------
Exploration Score:     0.0916 ± 0.0282 (95% CI: 0.0634-0.1198)
Exploitation Score:    0.3448 ± 0.0647 (95% CI: 0.2802-0.4095)
Exploration/Exploitation Ratio: 0.5718 ± 0.4314 (95% CI: 0.1405-1.0032)

PERFORMANCE METRICS:
----------------------------------------
Success Rate:          20.83% (95% CI: 9.24%-40.47%) [5/24]
Mean Completion Time:  10.1939 ± 0.7247 (95% CI: 9.4692-10.9185) ms
Mean RMSD:             270.9083 ± 42.6062 (95% CI: 228.3021-313.5145)



In [13]:
results = analyze_participant_session_task(merged_df, participant_id="MTRLRN070", session_no=5, task_type="Unimanual")


TRAJECTORY PERFORMANCE STATISTICS
Filters applied: Participant: MTRLRN070, Session: 5, Task: Unimanual
Total trials analyzed: 24

EXPLORATION & EXPLOITATION METRICS:
----------------------------------------
Exploration Score:     0.0153 ± 0.0054 (95% CI: 0.0099-0.0207)
Exploitation Score:    0.6404 ± 0.0926 (95% CI: 0.5478-0.7331)
Exploration/Exploitation Ratio: 0.0433 ± 0.0315 (95% CI: 0.0118-0.0748)

PERFORMANCE METRICS:
----------------------------------------
Success Rate:          100.00% (95% CI: 100.00%-100.00%) [24/24]
Mean Completion Time:  2.3605 ± 0.4659 (95% CI: 1.8946-2.8264) ms
Mean RMSD:             35.8417 ± 14.1163 (95% CI: 21.7254-49.9580)



In [16]:
results = analyze_participant_session_task(merged_df, participant_id="MTRLRN070", session_no=1, task_type="Unimanual")

TRAJECTORY PERFORMANCE STATISTICS
Filters applied: Participant: MTRLRN070, Session: 1, Task: Unimanual
Total trials analyzed: 24

EXPLORATION & EXPLOITATION METRICS:
----------------------------------------
Exploration Score:     0.0950 ± 0.0260 (95% CI: 0.0689-0.1210)
Exploitation Score:    0.3226 ± 0.0480 (95% CI: 0.2746-0.3706)
Exploration/Exploitation Ratio: 0.5579 ± 0.4252 (95% CI: 0.1328-0.9831)

PERFORMANCE METRICS:
----------------------------------------
Success Rate:          91.67% (95% CI: 74.15%-97.68%) [22/24]
Mean Completion Time:  7.3421 ± 1.0995 (95% CI: 6.2426-8.4416) ms
Mean RMSD:             152.7404 ± 23.7967 (95% CI: 128.9437-176.5371)

